# 14 — Eventi di gioco sulla lavagna 2D: possesso, passaggi, palle rubate, tiri

Estensione applicativa della tesi. Su una sequenza (SportsMOT oppure una clip registrata) il notebook:

1. **rileva** giocatori, palla, arbitri e `ball-in-basket` con il detector fine-tuned (RT-DETR-L, E1);
2. **traccia** i giocatori con BoT-SORT (E2) + interpolazione, e **pulisce** le tracce (doppioni, arbitro, cambi di squadra) come nel notebook 13;
3. classifica le **squadre** dal colore della maglia;
4. calcola l'**omografia** per frame e porta i giocatori sul campo;
5. segue la **palla** e stima il **possesso** frame per frame (classe `player-in-possession` del detector + vicinanza palla–giocatore, con isteresi);
6. ricava gli **eventi**:
   - **passaggio completato**: il possesso passa a un compagno;
   - **palla rubata / intercetto**: il possesso passa a un avversario senza un tiro in mezzo;
   - **tiro** (classi `player-jump-shot` / `player-layup-dunk`), **segnato** se compare `ball-in-basket` entro 3 s, altrimenti **sbagliato**; da 2 o da 3 punti in base alla distanza dal canestro;
   - **rimbalzo** (offensivo/difensivo) dopo un tiro sbagliato;
7. disegna la **lavagna tattica**: shot chart, frecce dei passaggi, palle rubate, video affiancato con mappa 2D animata;
8. **valuta** gli eventi contro un piccolo riferimento manuale (precisione e richiamo per tipo).

È un **prototipo**: le regole sono semplici e documentate, e i risultati vanno presentati con la loro valutazione.
Output in `events/<clip>/`. Runtime: **Colab GPU T4**. Serve il Secret `ROBOFLOW_API_KEY` (keypoint del campo).

In [ ]:
%pip install -q ultralytics trackers supervision inference "git+https://github.com/roboflow/sports.git@feat/basketball"
# dopo il pip: Runtime -> Riavvia sessione, poi esegui dalla cella successiva

In [ ]:
# Su Colab triton (usato solo per compilare modelli PyTorch, qui inutile) può far cadere il kernel all'import.
import sys
if 'triton' not in sys.modules: sys.modules['triton'] = None

# ==== SETUP COMUNE (uguale in tutti i notebook) ====
import sys, os
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Tesi_Basket')   # progetto su Drive
    FAST = Path('/content/fast')                                 # disco locale Colab (veloce)
else:
    ROOT = Path(r'F:\Tesi Basket')                               # progetto in locale
    FAST = ROOT / '_cache'
FAST.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Cartella del progetto non trovata: {ROOT}'
print('Progetto:', ROOT)

import faulthandler; faulthandler.enable()
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import json, time
import numpy as np, pandas as pd, cv2
import supervision as sv
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

## 1. Configurazione

In [ ]:
# Sorgente: nome di una sequenza SportsMOT (split val) oppure percorso di un video registrato
INPUT = 'v_5ekaksddqrc_c003'          # es. 'v_00HRwkvvjtQ_c001'  oppure  ROOT / 'videos' / 'mia_clip.mp4'
MAX_FRAMES = None                      # None = tutta la sequenza
WORK_W = 854                           # i frame vengono ridimensionati a questa larghezza (memoria e coerenza con le clip NBA)
DET_WEIGHTS = ROOT / 'models' / 'detector_best.pt'
DET_CONF = 0.05
COURT_MODEL_ID = 'basketball-court-detection-2/14'
HOMO_EVERY = 2                         # omografia ogni N frame (in mezzo si usa la più vicina)
FT_TO_M = 0.3048

# pulizia tracce (come notebook 13)
INTERP_MAX_GAP = 20; DUP_IOU = 0.7; SPLIT_MIN_SAMPLES = 5; OUTLIER_K = 3.0
SCOREBOARD_REL = None                  # SportsMOT: None. Clip NBA con tabellone in basso: (0.00, 0.79, 0.82, 0.95)

# palla e possesso
BALL_CONF = 0.25          # confidenza minima della palla
BALL_MAX_SPEED = 0.06     # spostamento massimo plausibile per frame (frazione della larghezza immagine)
BALL_INTERP = 8           # buchi della palla interpolati fino a N frame
POSS_CONF = 0.30          # confidenza minima della classe player-in-possession
POSS_RADIUS = 0.35        # distanza palla-box (in altezze della box) entro cui il giocatore "ha" la palla
POSS_MIN = 5              # frame consecutivi per confermare un nuovo possessore (isteresi)
POSS_LOST = 10            # frame senza possessore dopo i quali la palla è "libera"

# eventi
PASS_MAX_GAP_S = 2.0      # tempo massimo di volo del passaggio
SWITCH_DIST_M = 1.0       # cambio di ID "finto": il nuovo possessore è dove era il vecchio (stesso giocatore)
SHOT_MIN_FRAMES = 3       # frame con classe tiro per confermare un tiro
SHOT_WINDOW_S = 3.0       # finestra per vedere 'ball-in-basket' dopo il tiro
MADE_CONF = 0.30
EVENT_TOL_S = 1.0         # tolleranza temporale nel confronto con il riferimento manuale

CLIP_ID = Path(str(INPUT)).stem
OUT = ROOT / 'events' / CLIP_ID; OUT.mkdir(parents=True, exist_ok=True)
print('Clip:', CLIP_ID, '| output:', OUT)

## 2. Frame

In [ ]:
# ==== SportsMOT: download (val), ricerca sequenze e lista basket ====
import subprocess, urllib.request, configparser

SMOT_DIR = FAST / 'SportsMOT'
SMOT_CACHE = ROOT / 'datasets' / '_cache_sportsmot'     # zip conservati su Drive: niente ri-download
BASKET_LIST = ROOT / 'datasets' / 'sportsmot_basketball.txt'

def sportsmot_val_sequences():
    if not SMOT_DIR.exists() or not any(SMOT_DIR.rglob('gt.txt')):
        SMOT_CACHE.mkdir(parents=True, exist_ok=True)
        subprocess.run(['trackers', 'download', 'sportsmot', '--split', 'val',
                        '--output', str(SMOT_DIR), '--cache-dir', str(SMOT_CACHE)], check=True)
    seqs = sorted({p.parent.parent for p in SMOT_DIR.rglob('gt/gt.txt')})
    assert seqs, f'Nessuna sequenza trovata in {SMOT_DIR}'
    return seqs

def basketball_names():
    if not BASKET_LIST.is_file():
        url = 'https://huggingface.co/datasets/MCG-NJU/SportsMOT/resolve/main/splits_txt/basketball.txt'
        try:
            BASKET_LIST.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, BASKET_LIST)
        except Exception as e:
            raise RuntimeError(
                f'Non riesco a scaricare la lista basket ({e}).\n'
                f'Scaricala a mano da {url} (serve login Hugging Face) e salvala in {BASKET_LIST}')
    return set(BASKET_LIST.read_text().split())

def seq_info(seq):
    ini = seq / 'seqinfo.ini'
    fps, n = 25.0, len(list((seq / 'img1').glob('*.jpg')))
    if ini.is_file():
        c = configparser.ConfigParser(); c.read(ini)
        fps = float(c['Sequence'].get('frameRate', fps))
    return fps, n

def resize(f):
    return cv2.resize(f, (WORK_W, int(round(f.shape[0] * WORK_W / f.shape[1]))), interpolation=cv2.INTER_AREA) if f.shape[1] != WORK_W else f

frames = []
if str(INPUT).lower().endswith(('.mp4', '.mov', '.avi', '.mkv')):
    cap = cv2.VideoCapture(str(INPUT)); FPS = cap.get(cv2.CAP_PROP_FPS) or 25.0
    while True:
        ok, f = cap.read()
        if not ok or (MAX_FRAMES and len(frames) >= MAX_FRAMES): break
        frames.append(resize(f))
    cap.release()
else:
    seqs = {s.name: s for s in sportsmot_val_sequences()}
    assert INPUT in seqs, f'{INPUT} non è tra le sequenze val di SportsMOT'
    seq = seqs[INPUT]; FPS, _ = seq_info(seq)
    paths = sorted((seq / 'img1').glob('*.jpg'))[:MAX_FRAMES]
    frames = [resize(cv2.imread(str(p))) for p in tqdm(paths, desc='frame')]
N = len(frames); H_IMG, W_IMG = frames[0].shape[:2]
print(f'{N} frame {W_IMG}x{H_IMG} a {FPS:.2f} fps = {N / FPS:.1f} s')

## 3. Detection (tutte le classi, salvata in cache) e tracking dei giocatori

In [ ]:
from ultralytics import YOLO, RTDETR
import trackers as T
model = RTDETR(str(DET_WEIGHTS)) if 'rtdetr' in str(DET_WEIGHTS).lower() else YOLO(str(DET_WEIGHTS))
NAMES = model.names
print('classi:', NAMES)

cache = OUT / f'detections_w{WORK_W}.npy'
if cache.is_file() and len(np.unique(np.load(cache)[:, 0])) > 0:
    D = np.load(cache)
else:
    rows = []
    for fi, f in enumerate(tqdm(frames, desc='detection')):
        r = model.predict(f, conf=DET_CONF, iou=0.7, verbose=False)[0]      # NMS per classe: palla e giocatori non si eliminano
        for (x1, y1, x2, y2), c, k in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()):
            rows.append([fi, x1, y1, x2, y2, c, k])
    D = np.array(rows, np.float32).reshape(-1, 7); np.save(cache, D)
det = pd.DataFrame(D, columns=['frame', 'x1', 'y1', 'x2', 'y2', 'conf', 'cls'])
det['frame'] = det.frame.astype(int); det['name'] = det.cls.astype(int).map(NAMES).astype(str)
print(det[det.conf >= 0.3]['name'].value_counts().to_dict())

def iou_xyxy(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-9)

# BoT-SORT sulle classi giocatore; ogni box tracciata eredita la classe della detection con cui coincide
pl = det[det['name'].str.startswith('player')]
pl_by = dict(tuple(pl.groupby('frame')))
tracker = T.BoTSORTTracker(frame_rate=FPS)
rows = []
for fi in tqdm(range(N), desc='BoT-SORT'):
    g = pl_by.get(fi)
    xyxy = g[['x1', 'y1', 'x2', 'y2']].values.astype(np.float32) if g is not None else np.zeros((0, 4), np.float32)
    cf = g.conf.values.astype(np.float32) if g is not None else np.zeros(0, np.float32)
    tr = tracker.update(sv.Detections(xyxy=xyxy, confidence=cf, class_id=np.zeros(len(cf), int)), frame=frames[fi])
    if len(tr) == 0: continue
    M = iou_xyxy(tr.xyxy, xyxy) if len(xyxy) else np.zeros((len(tr), 0))
    for k, (b, tid) in enumerate(zip(tr.xyxy, tr.tracker_id)):
        if tid is None or tid < 0: continue
        j = int(M[k].argmax()) if M.shape[1] else -1
        ok = j >= 0 and M[k, j] > 0.5
        rows.append([fi, int(tid) + 1, *b, g['name'].values[j] if ok else 'player', float(cf[j]) if ok else np.nan])
tracks = pd.DataFrame(rows, columns=['frame', 'track_id', 'x1', 'y1', 'x2', 'y2', 'cls', 'cls_conf'])

def interpolate(t, max_gap, cols=('x1', 'y1', 'x2', 'y2')):
    out = []
    for tid, g in t.groupby('track_id'):
        g = g.set_index('frame').reindex(range(int(g.frame.min()), int(g.frame.max()) + 1))
        miss = g.x1.isna(); run = (miss != miss.shift()).cumsum()
        g[list(cols)] = g[list(cols)].interpolate(limit_area='inside')
        keep = ~miss | (miss.groupby(run).transform('sum') <= max_gap)
        g = g[keep].copy(); g['track_id'] = tid; g['cls'] = g['cls'].fillna('interp')
        g.index.name = 'frame'; out.append(g.reset_index())
    return pd.concat(out, ignore_index=True)
tracks = interpolate(tracks, INTERP_MAX_GAP)
tracks['frame'] = tracks.frame.astype(int); tracks['track_id'] = tracks.track_id.astype(int)
print('tracce:', tracks.track_id.nunique(), '| righe:', len(tracks))

## 4. Pulizia delle tracce e squadre (stesse regole del notebook 13)

In [ ]:
# doppioni: due box con IoU > DUP_IOU nello stesso frame = stesso giocatore; si tiene la traccia più lunga
length = tracks.groupby('track_id').size(); drop = []
for fi, g in tracks.groupby('frame'):
    if len(g) < 2: continue
    M = iou_xyxy(g[['x1', 'y1', 'x2', 'y2']].values, g[['x1', 'y1', 'x2', 'y2']].values); np.fill_diagonal(M, 0)
    ids, idx = g.track_id.values, g.index.values
    for i, j in zip(*np.where(np.triu(M) > DUP_IOU)):
        drop.append(idx[i] if length[ids[i]] < length[ids[j]] else idx[j])
tracks = tracks.drop(index=sorted(set(drop))).reset_index(drop=True)
tracks['foot_x'] = (tracks.x1 + tracks.x2) / 2; tracks['foot_y'] = tracks.y2
tracks['foot_ok'] = tracks.y2 < H_IMG - 3
if SCOREBOARD_REL:
    sx1, sy1, sx2, sy2 = SCOREBOARD_REL[0] * W_IMG, SCOREBOARD_REL[1] * H_IMG, SCOREBOARD_REL[2] * W_IMG, SCOREBOARD_REL[3] * H_IMG
    tracks.loc[tracks.foot_x.between(sx1, sx2) & tracks.y2.between(sy1 - 6, sy2), 'foot_ok'] = False
print(f'doppioni rimossi: {len(set(drop))} | piedi non affidabili: {(~tracks.foot_ok).sum()}')

# --- squadre: istogramma Lab della maglia, K-Means a 2 centri per traccia (numpy), squadra 0 = maglia scura ---
smp = np.concatenate([cv2.cvtColor(np.ascontiguousarray(f[int(.6*H_IMG):int(.78*H_IMG), int(.2*W_IMG):int(.8*W_IMG)]), cv2.COLOR_BGR2LAB).reshape(-1, 3)
                      for f in frames[::30]]).astype(np.float32)
COURT_LAB = np.median(smp, axis=0)

def jersey_feature(img, b):
    x1, y1, x2, y2 = map(int, b)
    x1, x2 = max(0, x1), min(W_IMG, x2); y1, y2 = max(0, y1), min(H_IMG, y2)
    w, h = x2 - x1, y2 - y1
    if h < 30 or w < 8: return None
    crop = np.ascontiguousarray(img[y1 + int(.15*h):y1 + int(.5*h), x1 + int(.2*w):x2 - int(.2*w)])
    if crop.size == 0 or crop.ndim != 3: return None
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float32)
    lab = lab[np.linalg.norm(lab - COURT_LAB, axis=1) > 25]
    if len(lab) < 20: return None
    hist = np.histogramdd(lab, bins=(4, 6, 6), range=((0, 256), (0, 256), (0, 256)))[0].ravel()
    return hist / hist.sum()

def sample_features(t):
    F, K, R = [], [], []
    for r in t[t.frame % 3 == 0].itertuples():
        f = jersey_feature(frames[int(r.frame)], (r.x1, r.y1, r.x2, r.y2))
        if f is not None: F.append(f); K.append(r.track_id); R.append(r.frame)
    return np.array(F), np.array(K), np.array(R)

class KMeans2:
    def __init__(self, n_init=20, iters=100, seed=0): self.n_init, self.iters, self.rng = n_init, iters, np.random.default_rng(seed)
    def fit(self, X):
        X = np.asarray(X, np.float64); best = None
        for _ in range(self.n_init):
            C = X[self.rng.choice(len(X), 2, replace=len(X) < 2)].copy()
            for _ in range(self.iters):
                lab = self._assign(X, C)
                newC = np.array([X[lab == k].mean(0) if (lab == k).any() else C[k] for k in range(2)])
                if np.allclose(newC, C): break
                C = newC
            inertia = ((X - C[self._assign(X, C)]) ** 2).sum()
            if best is None or inertia < best[0]: best = (inertia, C)
        self.cluster_centers_ = best[1]; return self
    @staticmethod
    def _assign(X, C): return np.argmin(((X[:, None, :] - C[None]) ** 2).sum(-1), axis=1)
    def predict(self, X): return self._assign(np.asarray(X, np.float64), self.cluster_centers_)

def fit_teams(F, K):
    med = {int(t): np.median(F[K == t], axis=0) for t in np.unique(K)}
    n = pd.Series(K).value_counts()
    fit_ids = [t for t in med if n[t] >= 5] or list(med)
    outliers = set()
    for _ in range(2):
        km = KMeans2().fit(np.array([med[t] for t in fit_ids if t not in outliers]))
        lum = [(c.reshape(4, 6, 6).sum((1, 2)) * np.array([32, 96, 160, 224])).sum() for c in km.cluster_centers_]
        if lum[0] > lum[1]: km.cluster_centers_ = km.cluster_centers_[::-1].copy()
        team = {t: int(km.predict(med[t][None])[0]) for t in med}
        d = pd.Series({t: np.linalg.norm(med[t] - km.cluster_centers_[team[t]]) for t in med})
        dl = d[[t for t in fit_ids if t not in outliers]]
        thr = dl.median() + OUTLIER_K * max((dl - dl.median()).abs().median() * 1.4826, 0.25 * dl.median(), 1e-6)
        outliers |= set(d.index[d > thr])
    for t in outliers: team[t] = -1
    return km, team

feats, keys, fr = sample_features(tracks)
km, team_of = fit_teams(feats, keys)
# divisione delle tracce che passano a un avversario (scambio di ID del tracker)
n_split = 0; lab_s = km.predict(feats); next_id = int(tracks.track_id.max()) + 1
for tid in np.unique(keys):
    if team_of[int(tid)] < 0: continue
    sel = np.argsort(fr[keys == tid]); f_t = fr[keys == tid][sel]; l_t = lab_s[keys == tid][sel]
    other = l_t != team_of[int(tid)]; run_start, cnt, cuts = None, 0, []
    for k, o in enumerate(other):
        if o: run_start = k if cnt == 0 else run_start; cnt += 1
        else:
            if cnt >= SPLIT_MIN_SAMPLES: cuts.append((f_t[run_start], f_t[k]))
            cnt = 0
    if cnt >= SPLIT_MIN_SAMPLES: cuts.append((f_t[run_start], N))
    for a, b in cuts:
        m = (tracks.track_id == tid) & (tracks.frame >= a) & (tracks.frame < b)
        tracks.loc[m, 'track_id'] = next_id; next_id += 1; n_split += 1
if n_split:
    feats, keys, fr = sample_features(tracks); km, team_of = fit_teams(feats, keys)
# arbitro: traccia che coincide spesso con un box della classe 'referee'
ref_by = {f: g[['x1', 'y1', 'x2', 'y2']].values for f, g in det[(det['name'] == 'referee') & (det.conf >= 0.3)].groupby('frame')}
for tid, g in tracks[tracks.frame % 3 == 0].groupby('track_id'):
    hits = [f in ref_by and iou_xyxy(np.array([[x1, y1, x2, y2]]), ref_by[f]).max() > 0.5
            for f, x1, y1, x2, y2 in g[['frame', 'x1', 'y1', 'x2', 'y2']].itertuples(index=False)]
    if hits and np.mean(hits) >= 0.3: team_of[int(tid)] = -1
tracks['team'] = tracks.track_id.map(team_of).fillna(-1).astype(int)
print('tracce divise:', n_split, '| tracce per squadra:', pd.Series(team_of).value_counts().to_dict(),
      '| escluse (arbitro/anomale):', sorted(t for t, k in team_of.items() if k < 0))
TEAM_COLORS = {0: (255, 140, 30), 1: (40, 40, 230), -1: (160, 160, 160)}   # BGR: 0 scura = blu, 1 chiara = rosso

fig, axes = plt.subplots(3, 1, figsize=(16, 7.5))
for ax, team in zip(axes, [0, 1, -1]):
    crops = []
    for tid in sorted(t for t, k in team_of.items() if k == team):
        g = tracks[tracks.track_id == tid]
        if not len(g): continue
        r = g.iloc[len(g) // 2]
        c = np.ascontiguousarray(frames[int(r.frame)][max(0, int(r.y1)):min(H_IMG, int(r.y2)), max(0, int(r.x1)):min(W_IMG, int(r.x2))])
        if c.size and c.shape[0] > 1 and c.shape[1] > 1:
            c = cv2.resize(c, (60, 120)); cv2.putText(c, str(tid), (2, 14), cv2.FONT_HERSHEY_SIMPLEX, .5, (0, 255, 255), 2); crops.append(c)
    if crops: ax.imshow(cv2.cvtColor(np.hstack(crops[:25]), cv2.COLOR_BGR2RGB))
    ax.set_title(f'Squadra {team}' if team >= 0 else 'Escluse (arbitro / anomale)'); ax.axis('off')
fig.tight_layout(); fig.savefig(OUT / 'controllo_squadre.png', dpi=130); plt.show()

## 5. Omografia e posizioni sul campo

In [ ]:
from sports import MeasurementUnit
from sports.basketball import CourtConfiguration, League, draw_court
court = CourtConfiguration(league=League.NBA, measurement_unit=MeasurementUnit.FEET)
V = np.asarray(court.vertices, dtype=np.float32)
L_FT, W_FT = float(court.court_length), float(court.court_width)

hfile = OUT / f'homographies_w{WORK_W}.json'
if hfile.is_file():
    hs = {int(k): np.asarray(v) for k, v in json.loads(hfile.read_text()).items()}
else:
    from inference import get_model
    try:
        from google.colab import userdata; API_KEY = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        API_KEY = os.environ.get('ROBOFLOW_API_KEY') or __import__('getpass').getpass('API key Roboflow: ')
    court_model = get_model(model_id=COURT_MODEL_ID, api_key=API_KEY)
    hs = {}
    for fi in tqdm(range(0, N, HOMO_EVERY), desc='omografia'):
        res = court_model.infer(frames[fi], confidence=0.3)[0]
        if not res.predictions: continue
        pr = max(res.predictions, key=lambda p: p.confidence)
        pts = [p for p in pr.keypoints if p.confidence >= 0.5 and 0 <= p.class_id < len(V)]
        if len(pts) >= 6:
            fwd, mask = cv2.findHomography(V[[p.class_id for p in pts]], np.float32([[p.x, p.y] for p in pts]), cv2.RANSAC, 8.0)
            if fwd is not None and mask is not None and mask.sum() >= 6:
                try: hs[fi] = np.linalg.inv(fwd)
                except np.linalg.LinAlgError: pass
    hfile.write_text(json.dumps({str(k): v.tolist() for k, v in hs.items()}))
keys_h = np.array(sorted(hs))
assert len(keys_h), 'Nessuna omografia valida: il campo non è abbastanza visibile in questa sequenza'
Hs = [hs[int(keys_h[np.abs(keys_h - fi).argmin()])] for fi in range(N)]     # omografia del frame calcolato più vicino
print(f'omografie valide: {len(hs)} su {len(range(0, N, HOMO_EVERY))} frame calcolati')

def to_court(x, y, fi):
    p = cv2.perspectiveTransform(np.float32([[[x, y]]]), Hs[int(fi)])[0, 0]
    return float(p[0]), float(p[1])

xy = [to_court(r.foot_x, r.foot_y, r.frame) if r.foot_ok else (np.nan, np.nan) for r in tracks.itertuples()]
tracks[['x_ft', 'y_ft']] = np.array(xy)
inside = tracks.x_ft.between(-3, L_FT + 3) & tracks.y_ft.between(-3, W_FT + 3)
tracks.loc[~inside, ['x_ft', 'y_ft']] = np.nan
W_SM = max(3, int(round(FPS / 2)) | 1)
tracks = tracks.sort_values(['track_id', 'frame']).reset_index(drop=True)
tracks[['xs_ft', 'ys_ft']] = tracks.groupby('track_id')[['x_ft', 'y_ft']].transform(
    lambda s: s.rolling(W_SM, center=True, min_periods=1).median().rolling(5, center=True, min_periods=1).mean())
tracks.to_csv(OUT / 'tracks_court.csv', index=False)
print('posizioni sul campo valide:', int(tracks.xs_ft.notna().sum()), 'su', len(tracks))

## 6. Palla e possesso

- **Palla**: per ogni frame la detection `ball` più sicura (≥ `BALL_CONF`); si scartano i salti impossibili (> `BALL_MAX_SPEED` × larghezza per frame) e si interpolano i buchi brevi.
- **Possesso grezzo** per frame: il giocatore con classe `player-in-possession` (se c'è), altrimenti il giocatore la cui box è più vicina alla palla (entro `POSS_RADIUS` altezze di box).
- **Isteresi**: un nuovo possessore viene confermato solo dopo `POSS_MIN` frame consecutivi; dopo `POSS_LOST` frame senza possessore la palla è "libera" (in volo o contesa).

In [ ]:
# --- palla ---
bd = det[(det['name'] == 'ball') & (det.conf >= BALL_CONF)].sort_values('conf', ascending=False).drop_duplicates('frame')
ball = pd.DataFrame(index=range(N), columns=['bx', 'by'], dtype=float)
last_f, last_xy = None, None
for r in bd.sort_values('frame').itertuples():
    c = np.array([(r.x1 + r.x2) / 2, (r.y1 + r.y2) / 2])
    if last_f is not None and r.frame - last_f <= 10 and np.linalg.norm(c - last_xy) > BALL_MAX_SPEED * W_IMG * (r.frame - last_f):
        continue                                       # salto impossibile: probabilmente un falso positivo
    ball.loc[r.frame] = c; last_f, last_xy = r.frame, c
ball_raw = ball.notna().all(1)
ball = ball.interpolate(limit=BALL_INTERP, limit_area='inside')
print(f'palla rilevata in {ball_raw.mean():.0%} dei frame, dopo interpolazione {ball.notna().all(1).mean():.0%}')

# --- possesso grezzo ---
by_frame = dict(tuple(tracks[tracks.team >= 0].groupby('frame')))
raw = np.full(N, -1)
for fi in range(N):
    g = by_frame.get(fi)
    if g is None: continue
    p = g[(g.cls == 'player-in-possession') & (g.cls_conf >= POSS_CONF)]
    bx, by = ball.loc[fi, 'bx'], ball.loc[fi, 'by']
    if len(p):
        if len(p) > 1 and not np.isnan(bx):
            p = p.assign(d=np.hypot((p.x1 + p.x2) / 2 - bx, (p.y1 + p.y2) / 2 - by)).sort_values('d')
        raw[fi] = int(p.track_id.iloc[0]); continue
    if np.isnan(bx): continue
    h = (g.y2 - g.y1).values
    dx = np.maximum.reduce([g.x1.values - bx, np.zeros(len(g)), bx - g.x2.values])
    dy = np.maximum.reduce([g.y1.values - by, np.zeros(len(g)), by - g.y2.values])
    d = np.hypot(dx, dy) / np.maximum(h, 1)
    k = int(d.argmin())
    if d[k] <= POSS_RADIUS: raw[fi] = int(g.track_id.values[k])

# --- isteresi ---
poss = np.full(N, -1); cur, cand, cnt, cand_start, lost = -1, -1, 0, 0, 0
for fi in range(N):
    r = raw[fi]
    if r == -1:
        lost += 1
        if lost > POSS_LOST: cur = -1
    else:
        lost = 0
        if r == cur: cand, cnt = -1, 0
        else:
            if r == cand: cnt += 1
            else: cand, cnt, cand_start = r, 1, fi
            if cnt >= POSS_MIN:
                cur = cand; poss[cand_start:fi] = cur; cand, cnt = -1, 0
    poss[fi] = cur

# segmenti di possesso
segs, s = [], None
for fi in range(N + 1):
    v = poss[fi] if fi < N else -2
    if s is not None and v != s[0]:
        segs.append({'track_id': s[0], 'start': s[1], 'end': fi - 1}); s = None
    if s is None and fi < N and v >= 0: s = (v, fi)
segs = pd.DataFrame(segs)
if len(segs):
    segs['team'] = segs.track_id.map(team_of).fillna(-1).astype(int)
    segs['dur_s'] = (segs.end - segs.start + 1) / FPS
print(f'possesso assegnato nel {(poss >= 0).mean():.0%} dei frame | segmenti di possesso: {len(segs)}')

## 7. Eventi

- **Tiro**: almeno `SHOT_MIN_FRAMES` frame della stessa traccia con classe `player-jump-shot` o `player-layup-dunk`; tiri a meno di 1,5 s l'uno dall'altro sono lo stesso tiro. **Segnato** se `ball-in-basket` compare entro `SHOT_WINDOW_S`; altrimenti **sbagliato**. **Da 3** se il tiratore è oltre l'arco (23,75 ft; 22 ft negli angoli).
- Tra due segmenti di possesso consecutivi (separati da al massimo `PASS_MAX_GAP_S`):
  - se in mezzo c'è un tiro → **rimbalzo** (offensivo se stessa squadra, difensivo se avversaria; nessun rimbalzo dopo un canestro);
  - stessa squadra, giocatore diverso → **passaggio completato** (escluso se il "nuovo" giocatore è nello stesso punto del vecchio: è un cambio di ID, non un passaggio);
  - squadra diversa → **palla rubata / intercetto**.

In [ ]:
def pos_at(tid, fi, win=15):
    g = tracks[(tracks.track_id == tid) & tracks.xs_ft.notna() & tracks.frame.between(fi - win, fi + win)]
    if not len(g): return (np.nan, np.nan)
    r = g.iloc[(g.frame - fi).abs().argmin()]
    return float(r.xs_ft), float(r.ys_ft)

BASKETS = [(5.25, W_FT / 2), (L_FT - 5.25, W_FT / 2)]
def shot_value(x, y):
    if np.isnan(x): return np.nan, np.nan
    bx, byy = min(BASKETS, key=lambda b: abs(b[0] - x))
    d = float(np.hypot(x - bx, y - byy))
    corner = abs(y - byy) > 22 and abs(x - bx) < 14
    return (3 if (d > 23.75 or (corner and d > 22)) else 2), d * FT_TO_M

# --- tiri ---
sh = tracks[tracks.cls.isin(['player-jump-shot', 'player-layup-dunk']) & (tracks.cls_conf >= 0.3) & (tracks.team >= 0)]
cand = []
for tid, g in sh.groupby('track_id'):
    f = g.frame.values; runs = np.split(f, np.where(np.diff(f) > 3)[0] + 1)
    for r in runs:
        if len(r) >= SHOT_MIN_FRAMES: cand.append((int(r[0]), int(tid), len(r)))
cand.sort()
shots = []
for f0, tid, n in cand:
    if shots and f0 - shots[-1]['frame'] < 1.5 * FPS:
        if n > shots[-1]['_n']: shots[-1].update(frame=f0, track_id=tid, _n=n)
        continue
    shots.append({'frame': f0, 'track_id': tid, '_n': n})
made_frames = det[(det['name'] == 'ball-in-basket') & (det.conf >= MADE_CONF)].frame.unique()
events = []
for s in shots:
    made = bool(((made_frames >= s['frame']) & (made_frames <= s['frame'] + SHOT_WINDOW_S * FPS)).any())
    x, y = pos_at(s['track_id'], s['frame']); val, dist = shot_value(x, y)
    events.append({'tipo': 'tiro_segnato' if made else 'tiro_sbagliato', 'frame': s['frame'], 'da': s['track_id'], 'a': np.nan,
                   'team': team_of.get(s['track_id'], -1), 'x_ft': x, 'y_ft': y, 'x2_ft': np.nan, 'y2_ft': np.nan,
                   'dettaglio': f'da {val} punti, {dist:.1f} m dal canestro' if not np.isnan(dist) else ''})
shot_frames = np.array([s['frame'] for s in shots])
shot_made = {s['frame']: e['tipo'] == 'tiro_segnato' for s, e in zip(shots, events)}

# --- transizioni di possesso ---
valid = segs[segs.team >= 0].reset_index(drop=True) if len(segs) else segs
for i in range(len(valid) - 1):
    a, b = valid.iloc[i], valid.iloc[i + 1]
    gap = (b.start - a.end) / FPS
    if gap > PASS_MAX_GAP_S or a.track_id == b.track_id: continue
    xa, ya = pos_at(a.track_id, a.end); xb, yb = pos_at(b.track_id, b.start)
    between = shot_frames[(shot_frames >= a.start) & (shot_frames <= b.start)]
    base = {'frame': int(b.start), 'da': int(a.track_id), 'a': int(b.track_id), 'team': int(b.team),
            'x_ft': xa, 'y_ft': ya, 'x2_ft': xb, 'y2_ft': yb}
    if len(between):
        if not shot_made[int(between[-1])]:
            events.append({**base, 'tipo': 'rimbalzo', 'dettaglio': 'offensivo' if a.team == b.team else 'difensivo'})
        continue
    if a.team == b.team:
        if gap < 0.3 and not np.isnan(xa) and np.hypot(xa - xb, ya - yb) * FT_TO_M < SWITCH_DIST_M:
            continue                                          # stesso giocatore con un nuovo ID, non un passaggio
        d = np.hypot(xa - xb, ya - yb) * FT_TO_M
        events.append({**base, 'frame': int(a.end), 'tipo': 'passaggio', 'dettaglio': f'{d:.1f} m' if not np.isnan(d) else ''})
    else:
        events.append({**base, 'tipo': 'palla_rubata', 'dettaglio': f'squadra {int(a.team)} -> {int(b.team)}'})

ev = pd.DataFrame(events, columns=['tipo', 'frame', 'da', 'a', 'team', 'x_ft', 'y_ft', 'x2_ft', 'y2_ft', 'dettaglio'])
ev = ev.sort_values('frame').reset_index(drop=True)
ev.insert(2, 'tempo_s', (ev.frame / FPS).round(2))
ev.insert(0, 'clip', CLIP_ID)
ev['verificato'] = ''          # da compilare a mano guardando il video: 1 = corretto, 0 = sbagliato
ev.to_csv(OUT / 'eventi.csv', index=False)
print(ev.tipo.value_counts().to_dict())
ev.drop(columns=['clip', 'x2_ft', 'y2_ft']).round(1)

## 8. Lavagna tattica: shot chart, passaggi, palle rubate e linea del tempo

In [ ]:
SCALE, PAD = 10, 30
base = draw_court(config=court, scale=SCALE, padding=PAD)
def to_map(x, y): return int(x * SCALE) + PAD, int(y * SCALE) + PAD
TEAM_RGB = {0: '#1e8cff', 1: '#e62828', -1: '#999999'}

fig, ax = plt.subplots(figsize=(13, 7.5))
ax.imshow(cv2.cvtColor(base, cv2.COLOR_BGR2RGB))
for r in ev.itertuples():
    if np.isnan(r.x_ft): continue
    x, y = to_map(r.x_ft, r.y_ft); c = TEAM_RGB.get(int(r.team), '#999')
    if r.tipo == 'passaggio' and not np.isnan(r.x2_ft):
        ax.annotate('', xy=to_map(r.x2_ft, r.y2_ft), xytext=(x, y), arrowprops=dict(arrowstyle='->', color=c, lw=2, alpha=.85))
    elif r.tipo == 'palla_rubata':
        xx, yy = to_map(r.x2_ft, r.y2_ft) if not np.isnan(r.x2_ft) else (x, y)
        ax.scatter(xx, yy, marker='*', s=420, color='gold', edgecolor=c, linewidth=2, zorder=5)
    elif r.tipo == 'tiro_segnato':
        ax.scatter(x, y, s=260, color='#2ca02c', edgecolor=c, linewidth=3, zorder=6)
    elif r.tipo == 'tiro_sbagliato':
        ax.scatter(x, y, s=260, marker='X', color='#d62728', edgecolor=c, linewidth=1.5, zorder=6)
    elif r.tipo == 'rimbalzo':
        ax.scatter(x, y, s=120, marker='s', color='white', edgecolor=c, linewidth=2, zorder=4)
from matplotlib.lines import Line2D
leg = [Line2D([], [], color=TEAM_RGB[0], lw=2, label='passaggio squadra 0 (scura)'),
       Line2D([], [], color=TEAM_RGB[1], lw=2, label='passaggio squadra 1 (chiara)'),
       Line2D([], [], marker='o', ls='', color='#2ca02c', ms=12, label='tiro segnato'),
       Line2D([], [], marker='X', ls='', color='#d62728', ms=12, label='tiro sbagliato'),
       Line2D([], [], marker='*', ls='', color='gold', ms=16, label='palla rubata'),
       Line2D([], [], marker='s', ls='', mfc='white', mec='k', ms=9, label='rimbalzo')]
ax.legend(handles=leg, loc='upper center', bbox_to_anchor=(.5, -.02), ncol=3, frameon=False)
ax.set_title(f'{CLIP_ID} — eventi rilevati automaticamente'); ax.axis('off')
fig.tight_layout(); fig.savefig(OUT / 'lavagna_eventi.png', dpi=180); plt.show()

# linea del tempo: possesso per squadra + eventi
fig, ax = plt.subplots(figsize=(14, 2.6))
for r in segs.itertuples() if len(segs) else []:
    ax.barh(0, (r.end - r.start + 1) / FPS, left=r.start / FPS, color=TEAM_RGB.get(int(r.team), '#999'), height=.5)
mk = {'passaggio': ('^', 'k'), 'palla_rubata': ('*', 'gold'), 'tiro_segnato': ('o', '#2ca02c'), 'tiro_sbagliato': ('X', '#d62728'), 'rimbalzo': ('s', 'w')}
for t, (m, c) in mk.items():
    e = ev[ev.tipo == t]
    if len(e): ax.scatter(e.tempo_s, np.full(len(e), .55), marker=m, color=c, edgecolor='k', s=90, label=t, zorder=5)
ax.set_yticks([]); ax.set_xlabel('tempo (s)'); ax.set_xlim(0, N / FPS); ax.set_ylim(-.4, .9)
ax.set_title('Possesso (blu = squadra 0, rosso = squadra 1) ed eventi'); ax.legend(ncol=5, loc='upper center', bbox_to_anchor=(.5, -.35), frameon=False)
fig.tight_layout(); fig.savefig(OUT / 'linea_del_tempo.png', dpi=160); plt.show()

## 9. Video: partita + lavagna animata

In [ ]:
mh = H_IMG; mw = int(base.shape[1] * mh / base.shape[0]); out_w = W_IMG + mw; out_w += out_w % 2
vw = cv2.VideoWriter(str(OUT / 'eventi_raw.mp4'), cv2.VideoWriter_fourcc(*'mp4v'), FPS, (out_w, H_IMG))
trk_by = dict(tuple(tracks.groupby('frame')))
LABEL = {'passaggio': 'PASSAGGIO', 'palla_rubata': 'PALLA RUBATA', 'tiro_segnato': 'TIRO SEGNATO', 'tiro_sbagliato': 'TIRO SBAGLIATO', 'rimbalzo': 'RIMBALZO'}
for fi in tqdm(range(N), desc='render'):
    img = frames[fi].copy(); mp = base.copy()
    for r in ev[(ev.frame <= fi) & (ev.tipo.str.startswith('tiro'))].itertuples():      # tiri: restano sulla lavagna
        if np.isnan(r.x_ft): continue
        p = to_map(r.x_ft, r.y_ft)
        if r.tipo == 'tiro_segnato': cv2.circle(mp, p, 10, (44, 160, 44), 3)
        else: cv2.drawMarker(mp, p, (40, 39, 214), cv2.MARKER_TILTED_CROSS, 20, 3)
    for r in ev[(ev.frame <= fi) & (ev.frame > fi - 2 * FPS) & ev.tipo.isin(['passaggio', 'palla_rubata'])].itertuples():
        if np.isnan(r.x_ft) or np.isnan(r.x2_ft): continue
        c = TEAM_COLORS[int(r.team)] if r.tipo == 'passaggio' else (0, 215, 255)
        cv2.arrowedLine(mp, to_map(r.x_ft, r.y_ft), to_map(r.x2_ft, r.y2_ft), c, 3, tipLength=.08)
    holder = poss[fi]
    for r in trk_by.get(fi, pd.DataFrame()).itertuples():
        c = TEAM_COLORS.get(int(r.team), (160, 160, 160)); th = 4 if r.track_id == holder else 2
        cv2.rectangle(img, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), c, th)
        cv2.putText(img, str(r.track_id), (int(r.x1), max(12, int(r.y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, .5, c, 2)
        if not np.isnan(r.xs_ft):
            p = to_map(r.xs_ft, r.ys_ft)
            cv2.circle(mp, p, 9, c, -1); cv2.circle(mp, p, 9, (0, 0, 0), 1)
            if r.track_id == holder: cv2.circle(mp, p, 14, (0, 200, 255), 3)          # anello giallo = possesso
    if not np.isnan(ball.loc[fi, 'bx']):
        cv2.circle(img, (int(ball.loc[fi, 'bx']), int(ball.loc[fi, 'by'])), 7, (0, 200, 255), 2)
    recent = ev[(ev.frame <= fi) & (ev.frame > fi - 1.5 * FPS)]
    for k, r in enumerate(recent.itertuples()):
        txt = LABEL[r.tipo] + (f'  {int(r.da)}->{int(r.a)}' if r.tipo in ('passaggio', 'palla_rubata') else f'  #{int(r.da)}  {r.dettaglio}')
        cv2.putText(img, txt, (12, 30 + 26 * k), cv2.FONT_HERSHEY_SIMPLEX, .7, (0, 0, 0), 4)
        cv2.putText(img, txt, (12, 30 + 26 * k), cv2.FONT_HERSHEY_SIMPLEX, .7, (255, 255, 255), 2)
    cv2.putText(mp, f'{fi / FPS:5.1f} s', (10, 22), cv2.FONT_HERSHEY_SIMPLEX, .6, (255, 255, 255), 2)
    canvas = np.zeros((H_IMG, out_w, 3), np.uint8); canvas[:, :W_IMG] = img; canvas[:, W_IMG:W_IMG + mw] = cv2.resize(mp, (mw, mh))
    vw.write(canvas)
vw.release()
import subprocess
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(OUT / 'eventi_raw.mp4'), '-c:v', 'libx264', '-crf', '23',
                '-pix_fmt', 'yuv420p', str(OUT / 'eventi.mp4')], check=True)
(OUT / 'eventi_raw.mp4').unlink()
print('Video:', OUT / 'eventi.mp4')

## 10. Valutazione contro un riferimento manuale

Guarda il video **originale** (non quello con le box, per non farti influenzare) e scrivi in `annotations/eventi_riferimento.csv` una riga per ogni evento vero:

```
clip,tipo,tempo_s,note
v_5ekaksddqrc_c003,passaggio,3.2,
v_5ekaksddqrc_c003,tiro_sbagliato,12.8,da 3 dall'angolo
```

Tipi: `passaggio`, `palla_rubata`, `tiro_segnato`, `tiro_sbagliato`, `rimbalzo`. Il tempo è quello del momento in cui il passaggio parte, il tiro viene preso, la palla viene rubata o presa a rimbalzo (bastano ±1 s).
Un evento rilevato è corretto se ce n'è uno vero dello stesso tipo entro `EVENT_TOL_S`; ogni evento vero si può usare una volta sola. Per i tiri si valuta anche il solo **tentativo di tiro** (segnato o sbagliato insieme).

In [ ]:
REF = ROOT / 'annotations' / 'eventi_riferimento.csv'
if not REF.is_file():
    pd.DataFrame(columns=['clip', 'tipo', 'tempo_s', 'note']).to_csv(REF, index=False)
    print('Creato il modello vuoto:', REF, '-> compilalo guardando il video e riesegui questa cella')
ref = pd.read_csv(REF)
ref = ref[ref['clip'] == CLIP_ID]

def match(pred_t, ref_t, tol):
    pairs = sorted((abs(p - r), i, j) for i, p in enumerate(pred_t) for j, r in enumerate(ref_t) if abs(p - r) <= tol)
    used_p, used_r = set(), set()
    for _, i, j in pairs:
        if i not in used_p and j not in used_r: used_p.add(i); used_r.add(j)
    return len(used_p)

if len(ref):
    rows = []
    groups = {t: [t] for t in ['passaggio', 'palla_rubata', 'tiro_segnato', 'tiro_sbagliato', 'rimbalzo']}
    groups['tiro (tentativo)'] = ['tiro_segnato', 'tiro_sbagliato']
    for name, types in groups.items():
        p = ev[ev.tipo.isin(types)].tempo_s.values; r = ref[ref.tipo.isin(types)].tempo_s.values
        tp = match(p, r, EVENT_TOL_S)
        rows.append({'tipo': name, 'rilevati': len(p), 'veri': len(r), 'corretti': tp,
                     'precisione': tp / len(p) if len(p) else np.nan, 'richiamo': tp / len(r) if len(r) else np.nan})
    val = pd.DataFrame(rows).round(2); val.insert(0, 'clip', CLIP_ID)
    val.to_csv(OUT / 'valutazione_eventi.csv', index=False); display(val)
else:
    print(f'Nessun evento di riferimento per {CLIP_ID} in {REF.name}: la valutazione verrà fatta quando lo compili.')

## 11. Riepilogo su tutte le clip elaborate

In [ ]:
allv = [pd.read_csv(p) for p in (ROOT / 'events').glob('*/valutazione_eventi.csv')]
if allv:
    v = pd.concat(allv)
    tot = v.groupby('tipo')[['rilevati', 'veri', 'corretti']].sum()
    tot['precisione'] = (tot.corretti / tot.rilevati).round(2); tot['richiamo'] = (tot.corretti / tot.veri).round(2)
    print('Clip valutate:', sorted(v['clip'].unique())); display(tot)
    tot.to_csv(ROOT / 'events' / 'valutazione_totale.csv')
else:
    print('Ancora nessuna clip valutata.')